# Medical Necessity Assessment - service logic demonstration

Demonstrates the end-to-end request-to-response flow that a Databricks-hosted scoring service
performs for a non-emergent ground transport order. A JSON payload representing a transport
order is received, the order-time clinical documentation is assessed against CMS criteria by a
language model served from Databricks Model Serving, and a structured determination is returned
to the caller.

No REST endpoint is created here. The payload is static and the response is printed. This
notebook establishes the logic that a serving endpoint would later wrap unchanged.

## Flow

1. Static request payload arrives (Section 3)
2. Request contract validation and scope check (Section 4)
3. CMS criteria supplied to the model as cited rule cards (Section 5)
4. Model extracts documented facts as strict JSON with verbatim evidence (Section 6)
5. Response validated, including evidence grounding against the source text (Section 7)
6. Deterministic rules assign the necessity class and GY disposition (Section 8)
7. Response envelope assembled and returned (Section 9)
8. Static payloads executed and results displayed (Sections 10-11)

## Design carried forward from the prior analysis notebooks

Extraction and determination are kept separate. The model reports what the documentation
states; a deterministic rule block assigns the class. Business rules can therefore change
without re-running the model, and the determination is reproducible and auditable.

Every criterion recorded as met must carry a verbatim quote from the submitted clinical text.
Quotes are verified against the source string, so an unsupported determination fails validation
rather than reaching the caller.

Read-only throughout. No table writes.

## 1. Configuration

Environment-specific values are isolated here so the rest of the notebook is portable between
the Databricks workspace and a local copy.

In [ ]:
import os
import re
import json
import time
import datetime
from typing import Dict, Any, List, Tuple

import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 160)

WORKSPACE_BASE_URL = "https://adb-2790612761746757.17.azuredatabricks.net/serving-endpoints"
LLM_MODEL = "databricks-gpt-oss-120b"

SERVICE_NAME = "med-nec-assessment"
SCHEMA_VERSION = "1.0"
CLASSIFICATION_METHOD = "llm_extraction + deterministic_rules"

LLM_TEMPERATURE = 0.0
LLM_MAX_TOKENS = 2000
LLM_RETRIES = 3

# Level of service ranking, used to compare what was requested against what the documentation
# supports. NONE means no medical transport is established by the documentation.
LOS_RANK = {"NONE": 0, "WHEELCHAIR": 1, "BLS": 2, "ALS": 3, "SCT": 4}

# Level of service codes outside the scope of this assessment. Emergent transports, fixed-wing
# quotes, wheelchair van, and organ procurement are excluded before assessment so the
# denominator reflects non-emergent ground orders only.
OUT_OF_SCOPE_LOS = {"EMG", "FWQUOTE", "WC", "ORGAN"}

RUN_ID = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")

print(f"service            : {SERVICE_NAME}")
print(f"schema_version     : {SCHEMA_VERSION}")
print(f"model              : {LLM_MODEL}")
print(f"run_id             : {RUN_ID}")

## 2. Model client

Identical invocation pattern to the nurse navigation notebook: the OpenAI client is pointed at
the Databricks serving-endpoints base URL and authenticated with the notebook context token.
When the logic is deployed behind an endpoint this block is the only part that changes, and it
changes to a service principal token rather than a notebook token.

In [ ]:
from openai import OpenAI

DATABRICKS_TOKEN = (
    dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
    if "dbutils" in dir() else os.environ.get("DATABRICKS_TOKEN", "")
)

client = OpenAI(api_key=DATABRICKS_TOKEN, base_url=WORKSPACE_BASE_URL)


def llm_call(system_prompt: str, messages: List[Dict[str, str]],
             max_tokens: int = LLM_MAX_TOKENS) -> str:
    """Call the served model and return the raw text content. Retries on transport errors."""
    payload = [{"role": "system", "content": system_prompt}] + messages
    last_error = None
    for attempt in range(LLM_RETRIES):
        try:
            resp = client.chat.completions.create(
                model=LLM_MODEL,
                messages=payload,
                temperature=LLM_TEMPERATURE,
                max_tokens=max_tokens,
            )
            content = resp.choices[0].message.content
            if isinstance(content, list):
                for item in content:
                    if isinstance(item, dict) and item.get("type") == "text":
                        return item.get("text", "")
                return json.dumps(content)
            return content
        except Exception as e:
            last_error = e
            wait = 2 ** attempt
            print(f"  model call retry {attempt + 1} ({e}); waiting {wait}s")
            time.sleep(wait)
    raise RuntimeError(f"model call failed after {LLM_RETRIES} attempts: {last_error}")

## 3. Static request payload

The payload mirrors the fields a transport order carries in TripMaster, so the contract
demonstrated here matches the data the production caller would hold at order time. The
clinical_data field is the order-time free text; questionnaire_responses are the structured
answers captured alongside it.

Five orders are included, chosen to exercise each determination path: documentation that
establishes both axes, non-specific documentation, absent documentation, monitoring established
without mobility, and a partial case.

In [ ]:
REQUEST_PAYLOADS = [
    {
        "order_id": "ORD-100418",
        "customer": "Texas Health Resources",
        "facility": "THR Presbyterian Dallas",
        "requested_level_of_service": "BLS",
        "emergent": False,
        "payer_type": "Medicare",
        "transport_reason": "Discharge to skilled nursing facility",
        "origin": "Acute care hospital",
        "destination": "Skilled nursing facility",
        "requested_datetime": "2026-08-05T14:20:00Z",
        "clinical_data": (
            "Pt s/p CVA with dense left hemiparesis. Unable to bear weight or sit upright "
            "without support, must remain supine for transport. Requires two-person assist "
            "for all transfers. Foley in place."
        ),
        "questionnaire_responses": {
            "bed_confined": True,
            "can_sit_in_chair": False,
            "ambulatory": False,
            "oxygen_lpm": 0,
        },
    },
    {
        "order_id": "ORD-100419",
        "customer": "MUSC Ground",
        "facility": "MUSC Charleston Main",
        "requested_level_of_service": "BLS",
        "emergent": False,
        "payer_type": "Medicare",
        "transport_reason": "Transfer to rehabilitation facility",
        "origin": "Acute care hospital",
        "destination": "Inpatient rehabilitation",
        "requested_datetime": "2026-08-05T15:05:00Z",
        "clinical_data": "Pt weak today, needs ambulance transport per protocol.",
        "questionnaire_responses": {
            "bed_confined": False,
            "can_sit_in_chair": True,
            "ambulatory": False,
            "oxygen_lpm": 0,
        },
    },
    {
        "order_id": "ORD-100420",
        "customer": "MUSC Ground",
        "facility": "MUSC Ashley River Tower",
        "requested_level_of_service": "BLS",
        "emergent": False,
        "payer_type": "Medicare",
        "transport_reason": "Return to residence",
        "origin": "Acute care hospital",
        "destination": "Private residence",
        "requested_datetime": "2026-08-05T16:40:00Z",
        "clinical_data": "",
        "questionnaire_responses": {},
    },
    {
        "order_id": "ORD-100421",
        "customer": "Texas Health Resources",
        "facility": "THR Harris Methodist Fort Worth",
        "requested_level_of_service": "ALS",
        "emergent": False,
        "payer_type": "Medicare",
        "transport_reason": "Transfer to long-term acute care",
        "origin": "Intensive care unit",
        "destination": "Long-term acute care hospital",
        "requested_datetime": "2026-08-06T09:15:00Z",
        "clinical_data": (
            "Ventilator dependent via tracheostomy, FiO2 40 percent, PEEP 5. Requires "
            "suctioning en route and continuous cardiac and pulse oximetry monitoring. "
            "Norepinephrine infusion running at 4 mcg/min. Non-ambulatory, unable to sit."
        ),
        "questionnaire_responses": {
            "bed_confined": True,
            "can_sit_in_chair": False,
            "ambulatory": False,
            "ventilator": True,
            "oxygen_lpm": 6,
        },
    },
    {
        "order_id": "ORD-100422",
        "customer": "Texas Health Resources",
        "facility": "THR Plano",
        "requested_level_of_service": "BLS",
        "emergent": False,
        "payer_type": "Medicare",
        "transport_reason": "Outpatient wound care appointment",
        "origin": "Skilled nursing facility",
        "destination": "Outpatient wound clinic",
        "requested_datetime": "2026-08-06T11:00:00Z",
        "clinical_data": (
            "Stage 3 sacral wound with drainage, dressing in place. Pt on 2L oxygen via "
            "nasal cannula. Transport to wound clinic."
        ),
        "questionnaire_responses": {
            "bed_confined": False,
            "can_sit_in_chair": True,
            "ambulatory": False,
            "oxygen_lpm": 2,
        },
    },
]

print(f"\nstatic payloads loaded: {len(REQUEST_PAYLOADS)}")

## 4. Request contract validation and scope check

Runs before any model call. Two purposes: reject malformed requests cheaply, and confirm the
order is within the non-emergent ground scope this assessment applies to. Rejecting an
out-of-scope order rather than assessing it is what prevented the scope defect corrected in the
batch analysis, where a silent filter changed the population being measured.

In [ ]:
REQUIRED_FIELDS = [
    "order_id", "requested_level_of_service", "emergent",
    "clinical_data", "customer",
]


def validate_request(payload: Dict[str, Any]) -> Tuple[bool, List[str]]:
    errors = []

    if not isinstance(payload, dict):
        return False, ["payload must be a JSON object"]

    for field in REQUIRED_FIELDS:
        if field not in payload:
            errors.append(f"missing required field: {field}")

    los = str(payload.get("requested_level_of_service", "")).upper()
    if los and los in OUT_OF_SCOPE_LOS:
        errors.append(f"level of service {los} is outside the non-emergent ground scope")
    if los and los not in LOS_RANK and los not in OUT_OF_SCOPE_LOS:
        errors.append(f"unrecognized level of service: {los}")

    if payload.get("emergent") is True:
        errors.append("emergent transports are outside the scope of this assessment")

    if not isinstance(payload.get("clinical_data", ""), str):
        errors.append("clinical_data must be a string")

    return (len(errors) == 0), errors

## 5. CMS criteria supplied to the model

The criteria are passed to the model as rule cards with stable identifiers. The model cites the
identifiers it relied on, which makes each determination traceable to the source text a reviewer
can open. In the batch analysis this role was filled by the concept dictionary; the same two
axes apply here.

Mobility axis: whether other means of transport are contraindicated (BPM10 10.2.1, 10.2.3)

Monitoring axis: whether the level of service is established (42 CFR 414.605)

In [ ]:
CMS_RULES = [
    ("BPM10_10_2_1",
     "Ambulance transport is covered where the beneficiary's condition is such that other "
     "means of transportation are contraindicated. The beneficiary's condition at the time of "
     "transport must be documented; a diagnosis alone does not establish necessity.",
     "Medicare Benefit Policy Manual Chapter 10, Section 10.2.1"),

    ("BPM10_10_2_3",
     "Bed confinement requires all three of the following: unable to get up from bed without "
     "assistance, unable to ambulate, and unable to sit in a chair or wheelchair. Bed "
     "confinement is one factor considered and is not by itself the sole criterion for medical "
     "necessity.",
     "Medicare Benefit Policy Manual Chapter 10, Section 10.2.3"),

    ("CFR_414_605_BLS",
     "Basic life support covers transport plus the provision of medically necessary supplies "
     "and services at the BLS level, including stretcher transport where the beneficiary "
     "cannot be safely transported seated.",
     "42 CFR 414.605"),

    ("CFR_414_605_ALS",
     "Advanced life support requires an ALS assessment by ALS personnel or the provision of at "
     "least one ALS intervention, such as medication administration by infusion, advanced "
     "airway management, or cardiac monitoring by qualified personnel.",
     "42 CFR 414.605"),

    ("CFR_414_605_SCT",
     "Specialty care transport is interfacility transport of a critically injured or ill "
     "beneficiary requiring ongoing care at a level beyond the scope of the EMT-Paramedic, "
     "such as ventilator management or continuous vasoactive infusion.",
     "42 CFR 414.605"),

    ("CFR_410_40_D_PCS",
     "Non-emergency scheduled transport requires a physician certification statement obtained "
     "at or before the time of the order, stating the condition that makes other transport "
     "contraindicated.",
     "42 CFR 410.40(d)"),

    ("INSUFFICIENT_TERMS",
     "Non-specific documentation such as general weakness, unsteady, deconditioned, needs "
     "transport, or per protocol does not establish medical necessity absent an underlying "
     "cause together with a specific functional deficit.",
     "BPM10 10.2.1 applied"),
]


def format_rule_cards() -> str:
    return "\n".join(f"[{rid}] {text} (Source: {src})" for rid, text, src in CMS_RULES)

## 6. Extraction prompt

The model reports only what the submitted documentation states, against the two axes. It makes
no determination and assigns no class. Every criterion recorded as met must carry a verbatim
quote taken from the clinical text.

In [ ]:
SYSTEM_PROMPT = f"""You are a documentation assessment assistant for NON-EMERGENT ground
ambulance transport orders. You assess whether the order-time clinical documentation states a
condition that meets CMS medical necessity criteria. You are not a clinical care advisor and
you do not decide what care the patient should receive.

TASK
Read the submitted transport order and report, against two axes, what the documentation
states. Do not assign a determination or a class. Report facts only.

AXIS 1 - MOBILITY. Whether the documentation states a condition making other means of
transport contraindicated. Governed by [BPM10_10_2_1] and [BPM10_10_2_3].

AXIS 2 - MONITORING. Whether the documentation states care requirements that establish a
level of service. Governed by [CFR_414_605_BLS], [CFR_414_605_ALS], [CFR_414_605_SCT].

CMS CRITERIA - decide using only these rules. Do not introduce policy that is not stated here.
{format_rule_cards()}

CONSTRAINTS
- Report only what the clinical text explicitly supports. Do not infer, and do not supply a
  clinical reason the documentation does not state.
- Handle negation correctly. "Denies shortness of breath" is not a respiratory finding.
- Every finding you report MUST carry a quote copied verbatim from the clinical text. Copy the
  characters exactly. Do not paraphrase, reorder, or correct a quote.
- If the clinical text is empty or contains no clinical content, set
  documentation_present to false and return empty findings arrays.
- Non-specific terms alone do not establish an axis. If the text contains only such terms, set
  insufficient_terms_only to true and cite [INSUFFICIENT_TERMS].
- supported_level_of_service is the highest level the documentation establishes, which may be
  lower than the level requested. Use NONE when the documentation establishes no medical
  transport requirement.
- The questionnaire responses are supporting context. A questionnaire flag alone, without
  clinical text stating the condition, does not establish a criterion.

ABBREVIATIONS: CVA=cerebrovascular accident, SNF=skilled nursing facility, LTACH=long-term
acute care hospital, ALS/BLS=advanced/basic life support, SCT=specialty care transport,
NC=nasal cannula, PEEP=positive end-expiratory pressure, s/p=status post.

OUTPUT: valid JSON only. No prose, no markdown fences. All keys present. Booleans never null.

SCHEMA:
{{
  "order_id": string,
  "documentation_present": boolean,
  "insufficient_terms_only": boolean,

  "mobility_axis": {{
    "criterion_met": boolean,
    "findings": [
      {{"concept": string, "quote": string, "cms_rule_id": string}}
    ],
    "bed_confined_prongs": {{
      "unable_to_get_up": boolean,
      "unable_to_ambulate": boolean,
      "unable_to_sit": boolean
    }}
  }},

  "monitoring_axis": {{
    "criterion_met": boolean,
    "supported_level_of_service": "NONE|WHEELCHAIR|BLS|ALS|SCT",
    "findings": [
      {{"concept": string, "quote": string, "cms_rule_id": string}}
    ]
  }},

  "missing_elements": [string],
  "clarifying_prompt": string,
  "cited_rules": [string],
  "reasoning": string
}}

FIELD NOTES
- concept: a short label for what the quote establishes, for example "unable to bear weight",
  "ventilator dependent", "continuous infusion".
- bed_confined_prongs: set each prong true only where the text states it. All three prongs are
  required for bed confinement under [BPM10_10_2_3], and bed confinement alone is not the sole
  criterion.
- missing_elements: what a reviewer would need added to the order for the criteria to be
  established. Empty when both axes are established.
- clarifying_prompt: one short, specific question that would let the ordering clinician correct
  the order at the point of entry. Empty string when nothing is missing.
- reasoning: three sentences at most, describing what the documentation states.
"""

Few-shot examples. Two cases, teaching the distinction that drives most of the volume:
non-specific text does not establish an axis, and specific functional deficit does.

In [ ]:
FEWSHOT = [
    {
        "role": "user",
        "content": json.dumps({
            "order_id": "EX-1",
            "requested_level_of_service": "BLS",
            "clinical_data": "Patient deconditioned, needs transport to SNF.",
            "questionnaire_responses": {"bed_confined": False, "ambulatory": False},
        }),
    },
    {
        "role": "assistant",
        "content": json.dumps({
            "order_id": "EX-1",
            "documentation_present": True,
            "insufficient_terms_only": True,
            "mobility_axis": {
                "criterion_met": False,
                "findings": [],
                "bed_confined_prongs": {
                    "unable_to_get_up": False,
                    "unable_to_ambulate": False,
                    "unable_to_sit": False,
                },
            },
            "monitoring_axis": {
                "criterion_met": False,
                "supported_level_of_service": "NONE",
                "findings": [],
            },
            "missing_elements": [
                "underlying cause of the deconditioning",
                "specific functional deficit",
                "why the patient cannot be transported seated",
            ],
            "clarifying_prompt": (
                "What condition prevents the patient from sitting safely in a wheelchair or "
                "private vehicle? For example, unable to bear weight following a hip fracture."
            ),
            "cited_rules": ["INSUFFICIENT_TERMS", "BPM10_10_2_1"],
            "reasoning": (
                "The text states deconditioning without an underlying cause or a functional "
                "deficit. No condition contraindicating other transport is documented. No "
                "monitoring requirement is documented."
            ),
        }),
    },
    {
        "role": "user",
        "content": json.dumps({
            "order_id": "EX-2",
            "requested_level_of_service": "BLS",
            "clinical_data": (
                "Right hip fracture, non-weight bearing, unable to sit upright, requires "
                "supine positioning for transport."
            ),
            "questionnaire_responses": {"bed_confined": True, "can_sit_in_chair": False},
        }),
    },
    {
        "role": "assistant",
        "content": json.dumps({
            "order_id": "EX-2",
            "documentation_present": True,
            "insufficient_terms_only": False,
            "mobility_axis": {
                "criterion_met": True,
                "findings": [
                    {"concept": "unable to bear weight",
                     "quote": "non-weight bearing",
                     "cms_rule_id": "BPM10_10_2_3"},
                    {"concept": "unable to sit",
                     "quote": "unable to sit upright",
                     "cms_rule_id": "BPM10_10_2_3"},
                    {"concept": "requires supine positioning",
                     "quote": "requires supine positioning for transport",
                     "cms_rule_id": "BPM10_10_2_1"},
                ],
                "bed_confined_prongs": {
                    "unable_to_get_up": False,
                    "unable_to_ambulate": True,
                    "unable_to_sit": True,
                },
            },
            "monitoring_axis": {
                "criterion_met": True,
                "supported_level_of_service": "BLS",
                "findings": [
                    {"concept": "stretcher transport required",
                     "quote": "requires supine positioning for transport",
                     "cms_rule_id": "CFR_414_605_BLS"},
                ],
            },
            "missing_elements": [],
            "clarifying_prompt": "",
            "cited_rules": ["BPM10_10_2_1", "BPM10_10_2_3", "CFR_414_605_BLS"],
            "reasoning": (
                "The text states a hip fracture with non-weight bearing status and inability "
                "to sit upright, which contraindicates seated transport. Supine positioning "
                "establishes stretcher transport at the BLS level."
            ),
        }),
    },
]


def build_user_message(payload: Dict[str, Any]) -> str:
    """Serialize the request into the message the model receives."""
    return json.dumps({
        "order_id": payload.get("order_id"),
        "requested_level_of_service": payload.get("requested_level_of_service"),
        "transport_reason": payload.get("transport_reason"),
        "origin": payload.get("origin"),
        "destination": payload.get("destination"),
        "clinical_data": payload.get("clinical_data", ""),
        "questionnaire_responses": payload.get("questionnaire_responses", {}),
    }, indent=2)

## 7. Response validation

Three gates, applied in order. Structural validation confirms the schema. Type validation
confirms the fields the determination depends on. Evidence grounding confirms every quote
appears in the submitted text, which is the check that stops an unsupported determination from
reaching the caller.

In [ ]:
REQUIRED_TOP_KEYS = {
    "order_id", "documentation_present", "insufficient_terms_only",
    "mobility_axis", "monitoring_axis", "missing_elements",
    "clarifying_prompt", "cited_rules", "reasoning",
}

VALID_LOS = set(LOS_RANK.keys())


def normalize_text(s: str) -> str:
    """Lowercase and collapse whitespace so quote matching tolerates formatting only."""
    return re.sub(r"\s+", " ", str(s)).strip().lower()


def strip_fences(raw: str) -> str:
    """Remove markdown fences if the model emits them despite the instruction."""
    txt = str(raw).strip()
    if txt.startswith("```"):
        txt = re.sub(r"^```[a-zA-Z]*\s*", "", txt)
        txt = re.sub(r"\s*```$", "", txt)
    return txt.strip()


def validate_response(raw: str, source_text: str) -> Tuple[bool, Dict[str, Any], List[str]]:
    problems = []

    try:
        obj = json.loads(strip_fences(raw))
    except Exception as e:
        return False, {}, [f"invalid JSON: {e}"]

    missing = REQUIRED_TOP_KEYS - set(obj)
    if missing:
        return False, obj, [f"missing keys: {sorted(missing)}"]

    for key in ["documentation_present", "insufficient_terms_only"]:
        if not isinstance(obj.get(key), bool):
            problems.append(f"{key} must be boolean")

    for axis in ["mobility_axis", "monitoring_axis"]:
        sec = obj.get(axis)
        if not isinstance(sec, dict):
            problems.append(f"{axis} must be an object")
            continue
        if not isinstance(sec.get("criterion_met"), bool):
            problems.append(f"{axis}.criterion_met must be boolean")
        if not isinstance(sec.get("findings"), list):
            problems.append(f"{axis}.findings must be an array")

    los = str(obj.get("monitoring_axis", {}).get("supported_level_of_service", "")).upper()
    if los not in VALID_LOS:
        problems.append(f"supported_level_of_service not recognized: {los}")

    prongs = obj.get("mobility_axis", {}).get("bed_confined_prongs", {})
    if not isinstance(prongs, dict):
        problems.append("bed_confined_prongs must be an object")
    else:
        for p in ["unable_to_get_up", "unable_to_ambulate", "unable_to_sit"]:
            if not isinstance(prongs.get(p), bool):
                problems.append(f"bed_confined_prongs.{p} must be boolean")

    # Evidence grounding: a criterion recorded as met requires at least one finding, and every
    # quote must appear in the submitted clinical text.
    src = normalize_text(source_text)
    for axis in ["mobility_axis", "monitoring_axis"]:
        sec = obj.get(axis, {})
        findings = sec.get("findings", []) if isinstance(sec, dict) else []
        if isinstance(sec, dict) and sec.get("criterion_met") and not findings:
            problems.append(f"{axis}.criterion_met is true with no supporting finding")
        for i, f in enumerate(findings if isinstance(findings, list) else []):
            if not isinstance(f, dict):
                problems.append(f"{axis}.findings[{i}] must be an object")
                continue
            quote = normalize_text(f.get("quote", ""))
            if not quote:
                problems.append(f"{axis}.findings[{i}] has no quote")
            elif quote not in src:
                problems.append(
                    f"{axis}.findings[{i}] quote not found in submitted text: "
                    f"{str(f.get('quote'))[:60]}"
                )

    return (len(problems) == 0), obj, problems

## 8. Determination

Deterministic. The model supplies facts; this block assigns the class. Three factual labels are
produced - necessary, not_necessary, indeterminate - together with the GY disposition the class
implies. Thresholds and the confidence formula are tunable modelling choices, not CMS figures,
and are subject to review by the medical necessity subject matter experts.

In [ ]:
GY_DISPOSITION = {
    "necessary": "documented reason present",
    "not_necessary": "no documented reason - GY candidate",
    "indeterminate": "partial - review",
}

CONFIDENCE_BASE = 0.50
CONFIDENCE_PER_FINDING = 0.10
CONFIDENCE_FINDING_CAP = 0.30
CONFIDENCE_BOTH_AXES = 0.10
CONFIDENCE_ALL_PRONGS = 0.09
CONFIDENCE_NO_DOCUMENTATION = 0.95
CONFIDENCE_NON_SPECIFIC = 0.25


def compute_confidence(obj: Dict[str, Any], necessity_class: str) -> float:
    """Confidence in the determination, not in the clinical picture. Absence of documentation
    is a determination made with high confidence, since nothing is open to interpretation."""
    if not obj.get("documentation_present", False):
        return CONFIDENCE_NO_DOCUMENTATION

    mob = obj.get("mobility_axis", {})
    mon = obj.get("monitoring_axis", {})
    n_findings = len(mob.get("findings", [])) + len(mon.get("findings", []))

    score = CONFIDENCE_BASE
    score += min(n_findings * CONFIDENCE_PER_FINDING, CONFIDENCE_FINDING_CAP)

    if mob.get("criterion_met") and mon.get("criterion_met"):
        score += CONFIDENCE_BOTH_AXES

    prongs = mob.get("bed_confined_prongs", {})
    if all(bool(prongs.get(p)) for p in
           ["unable_to_get_up", "unable_to_ambulate", "unable_to_sit"]):
        score += CONFIDENCE_ALL_PRONGS

    if necessity_class == "indeterminate":
        score -= 0.10

    if obj.get("insufficient_terms_only") and necessity_class == "not_necessary":
        score += CONFIDENCE_NON_SPECIFIC

    return round(min(max(score, 0.05), 0.99), 2)


def determine(obj: Dict[str, Any], requested_los: str) -> Dict[str, Any]:
    """Assign the necessity class, the level of service alignment, and the GY disposition."""
    mob_met = bool(obj.get("mobility_axis", {}).get("criterion_met"))
    mon_met = bool(obj.get("monitoring_axis", {}).get("criterion_met"))
    documented = bool(obj.get("documentation_present"))
    non_specific = bool(obj.get("insufficient_terms_only"))

    supported_los = str(
        obj.get("monitoring_axis", {}).get("supported_level_of_service", "NONE")
    ).upper()
    requested = str(requested_los).upper()

    req_rank = LOS_RANK.get(requested, 0)
    sup_rank = LOS_RANK.get(supported_los, 0)

    if sup_rank == req_rank:
        los_alignment = "aligned"
    elif sup_rank < req_rank:
        los_alignment = "requested level exceeds documentation"
    else:
        los_alignment = "documentation exceeds requested level"

    if not documented:
        necessity_class = "not_necessary"
        basis = "no clinical documentation submitted with the order"
    elif non_specific and not mob_met and not mon_met:
        necessity_class = "not_necessary"
        basis = "non-specific terms only; no condition contraindicating other transport stated"
    elif mob_met and sup_rank >= req_rank:
        necessity_class = "necessary"
        basis = "mobility criterion stated and the requested level of service is supported"
    elif mob_met and sup_rank < req_rank:
        necessity_class = "indeterminate"
        basis = "mobility criterion stated; requested level of service not supported by the text"
    elif mon_met and not mob_met:
        necessity_class = "indeterminate"
        basis = "monitoring requirement stated; mobility criterion not established"
    elif not mob_met and not mon_met:
        necessity_class = "not_necessary"
        basis = "documentation present but neither criterion is established"
    else:
        necessity_class = "indeterminate"
        basis = "documentation partially establishes the criteria"

    return {
        "necessity_class": necessity_class,
        "determination_basis": basis,
        "mobility_criterion_met": mob_met,
        "monitoring_criterion_met": mon_met,
        "requested_level_of_service": requested,
        "supported_level_of_service": supported_los,
        "level_of_service_alignment": los_alignment,
        "gy_disposition": GY_DISPOSITION[necessity_class],
        "confidence": compute_confidence(obj, necessity_class),
    }

## 9. Assessment function

The single callable a serving endpoint would wrap. Takes the request payload, returns the
response envelope. Everything above this point is configuration and helpers; everything below is
execution of the static payloads.

In [ ]:
def assess_order(payload: Dict[str, Any]) -> Dict[str, Any]:
    started = time.time()
    received = datetime.datetime.utcnow().isoformat() + "Z"

    def envelope(status: str, **extra) -> Dict[str, Any]:
        base = {
            "schema_version": SCHEMA_VERSION,
            "order_id": payload.get("order_id"),
            "received_utc": received,
            "status": status,
            "model": LLM_MODEL,
            "classification_method": CLASSIFICATION_METHOD,
            "latency_ms": int((time.time() - started) * 1000),
        }
        base.update(extra)
        return base

    ok, errors = validate_request(payload)
    if not ok:
        return envelope("rejected", errors=errors)

    try:
        raw = llm_call(SYSTEM_PROMPT, FEWSHOT + [
            {"role": "user", "content": build_user_message(payload)}
        ])
    except Exception as e:
        return envelope("error", errors=[f"model call failed: {e}"])

    valid, obj, problems = validate_response(raw, payload.get("clinical_data", ""))
    if not valid:
        return envelope("error", errors=problems, raw_response=raw[:1500])

    result = determine(obj, payload.get("requested_level_of_service", ""))

    findings = []
    for axis_key, axis_name in [("mobility_axis", "mobility"),
                                ("monitoring_axis", "monitoring")]:
        for f in obj.get(axis_key, {}).get("findings", []):
            findings.append({
                "axis": axis_name,
                "concept": f.get("concept"),
                "quote": f.get("quote"),
                "cms_rule_id": f.get("cms_rule_id"),
            })

    return envelope(
        "ok",
        customer=payload.get("customer"),
        documentation_present=bool(obj.get("documentation_present")),
        insufficient_terms_only=bool(obj.get("insufficient_terms_only")),
        bed_confined_prongs=obj.get("mobility_axis", {}).get("bed_confined_prongs", {}),
        evidence=findings,
        missing_elements=obj.get("missing_elements", []),
        clarifying_prompt=obj.get("clarifying_prompt", ""),
        cited_rules=obj.get("cited_rules", []),
        reasoning=obj.get("reasoning", ""),
        **result,
    )

## 10. Execution of the static payloads

Each request is printed as the caller would send it, and each response as the caller would
receive it. This is the demonstration: payload in, determination out.

In [ ]:
responses = []

for payload in REQUEST_PAYLOADS:
    print("\n" + "=" * 100)
    print(f"REQUEST  {payload['order_id']}   {payload['customer']}   "
          f"requested {payload['requested_level_of_service']}")
    print("-" * 100)
    print(json.dumps(payload, indent=2))

    response = assess_order(payload)
    responses.append(response)

    print("-" * 100)
    print(f"RESPONSE {payload['order_id']}   status={response['status']}   "
          f"{response.get('latency_ms')} ms")
    print("-" * 100)
    print(json.dumps(response, indent=2))

## 11. Result summary

One row per order, showing the fields a caller acts on: the class, the level of service
comparison, the GY disposition, and whether a correction prompt was returned to the ordering
clinician.

In [ ]:
summary = pd.DataFrame([
    {
        "order_id": r.get("order_id"),
        "customer": r.get("customer"),
        "status": r.get("status"),
        "necessity_class": r.get("necessity_class"),
        "mobility": r.get("mobility_criterion_met"),
        "monitoring": r.get("monitoring_criterion_met"),
        "requested_los": r.get("requested_level_of_service"),
        "supported_los": r.get("supported_level_of_service"),
        "los_alignment": r.get("level_of_service_alignment"),
        "gy_disposition": r.get("gy_disposition"),
        "confidence": r.get("confidence"),
        "evidence_count": len(r.get("evidence", []) or []),
        "clarifying_prompt_returned": bool(r.get("clarifying_prompt")),
        "latency_ms": r.get("latency_ms"),
    }
    for r in responses
])

print("\n" + "=" * 100)
print("SUMMARY")
print("=" * 100)
print(summary.to_string(index=False))

ok_rows = summary[summary["status"] == "ok"]
if len(ok_rows):
    print("\nlatency (ok responses)")
    print(f"  mean   {ok_rows['latency_ms'].mean():.0f} ms")
    print(f"  median {ok_rows['latency_ms'].median():.0f} ms")
    print(f"  max    {ok_rows['latency_ms'].max():.0f} ms")

print("\nclass distribution")
print(summary["necessity_class"].value_counts(dropna=False).to_string())

## 12. What the endpoint adds

The logic above is complete and unchanged by deployment. Serving it adds the following, none of
which alters the determination.

Transport. A Databricks Model Serving custom endpoint, or an Azure API Management front end,
exposing POST /assess. The request body is the payload in Section 3; the response body is the
envelope in Section 9.

Authentication. A service principal token rather than the notebook context token, with the
caller authenticated at the gateway.

Concurrency. Section 10 runs the payloads sequentially for readability. The endpoint handles
requests concurrently; observed single-request latency is the figure that matters for the
order-entry experience, since assessment occurs while the clinician is still in the form.

Logging. Each request and response written to a Delta table for audit, drift monitoring, and
subject matter expert review. This notebook writes nothing.

Versioning. SCHEMA_VERSION and the prompt revision returned in every response so a determination
can be reproduced against the logic in force when it was made.

## Limitations to carry into review

The assessment reads order-time documentation only. The final billing determination also depends
on the crew patient care report supporting the physician certification statement, and the
patient care report is held in ImageTrend, a separate source.

The confidence formula and the level of service comparison are modelling choices, not CMS
figures. Both require validation with the medical necessity subject matter experts before any
figure derived from them is reported.

Evidence grounding rejects a quote that does not appear in the submitted text. It does not
verify that the quote establishes the concept the model assigned to it. That judgement remains
with the reviewer.